# 17 · Alertas en tiempo real (capa de velocidad)

**Propósito:** emitir la alerta de cada vuelo **en el momento en que llega el METAR de T−2 h**, sin esperar al lote
mensual. Es la capa de velocidad de una arquitectura **Lambda**:

| Capa | Qué hace | Dónde |
|---|---|---|
| Lotes | ingesta mensual, silver/gold, **entrena** el modelo con toda la historia, KPI y valor | notebooks 01–16 |
| Velocidad | lee los METAR a medida que llegan, arma el clima vigente y **aplica** el modelo guardado | este notebook |
| Servicio | gold en S3 consultado con Athena por el tablero | `gold/alertas_tiempo_real`, `gold/kpi_*` |

**¿Por qué hace falta tiempo real aquí y no en el resto?** El jefe de turno del CCO decide a T−2 h. El METAR se
publica cada hora: un lote nocturno no ve el clima del día y, si la alerta llega tarde, la demora que se podía mitigar
(avión de reserva, tripulación, aviso a pasajeros) ya no se puede. Basta una latencia de **minutos** (micro-lotes);
pedir segundos (Kafka) solo agregaría costo. Entrenar, los KPI mensuales y el valor de negocio siguen por lotes:
esperar no les cuesta nada.

**¿Por qué Lambda y no Kappa?** El modelo se entrena con 3 años de historia; guardar 3 años en un log de eventos (Kafka
retiene 7 días por defecto) sería caro. Se usa una variante "Kappa ligera" solo para reprocesar: un checkpoint nuevo
con `availableNow` vuelve a leer la carpeta de llegadas (sección 7).

**Qué es real y qué es simulado:** los METAR y los vuelos son reales (2025, periodo de test que el modelo no vio). Lo
simulado es **el momento de llegada**: el notebook deja caer, uno por uno, los archivos de cada hora de un día pasado,
como si llegaran en vivo. Los despegues del día no llegan por el flujo: las variables de red y de historia del vuelo
se toman de gold como tabla estática (ya están calculadas con lo conocido a T−2 h).

**Entrada:** `silver/metar_obs` (texto METAR original), `gold/features_salida`, `gold/modelos/final_*`.

**Salida:** `gold/alertas_tiempo_real/run=<id>/` y evidencias en `docs/evidencias/tiempo_real/`.

**Requisito:** notebooks 06, 08 y 12.

In [ ]:
import os
from pathlib import Path

import yaml

# Carpeta del repositorio (este notebook está en notebooks/)
RAIZ = Path.cwd().parent
with open(RAIZ / "config" / "pipeline.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
tablas = cfg["lake"]["tablas"]

# En la EC2 la variable LAKE_BUCKET trae el nombre del bucket; en una laptop se trabaja sobre data/lake
BUCKET = os.environ.get("LAKE_BUCKET", "").strip()
RAIZ_LOCAL = RAIZ / cfg["lake"]["raiz_local"]
BASE = f"s3a://{BUCKET}" if BUCKET else RAIZ_LOCAL.resolve().as_uri()


def ruta(tabla: str) -> str:
    """Ruta completa de una tabla declarada en config/pipeline.yaml."""
    return f"{BASE}/{tablas[tabla]}"


def meses(desde: str, hasta: str) -> list[tuple[int, int]]:
    """Pares (año, mes) entre dos fechas AAAA-MM, ambas incluidas."""
    anio, mes = map(int, desde.split("-"))
    anio_fin, mes_fin = map(int, hasta.split("-"))
    salida = []
    while (anio, mes) <= (anio_fin, mes_fin):
        salida.append((anio, mes))
        anio, mes = (anio + 1, 1) if mes == 12 else (anio, mes + 1)
    return salida


DESDE, HASTA = cfg["periodo"]["inicio"], cfg["periodo"]["fin"]
print("Destino:", f"s3://{BUCKET}" if BUCKET else "data/lake (local)")
print("Periodo:", DESDE, "a", HASTA)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
import csv
import io
import time
from datetime import UTC, datetime

import matplotlib.pyplot as plt
import pandas as pd
from pyspark.errors import AnalysisException
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array

# En la EC2 Spark toma la configuración del clúster (spark-defaults.conf); en una laptop corre en modo local
constructor = SparkSession.builder.appName("latam-bigdata-tiempo-real")
if not BUCKET:
    constructor = (
        constructor.master("local[*]")
        .config("spark.sql.shuffle.partitions", str(cfg["spark"]["particiones_shuffle_local"]))
        .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "3g"))
    )
spark = (
    constructor.config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")


def en_periodo(df, col_anio="year", col_mes="month"):
    """Deja solo las filas cuyo año y mes están dentro del periodo de config/pipeline.yaml."""
    ym = F.format_string("%04d-%02d", F.col(col_anio), F.col(col_mes))
    return df.where((ym >= DESDE) & (ym <= HASTA))

In [ ]:
ST = cfg["streaming"]
# ----- Parámetros de la demostración -----
MODO = "continuo"            # "continuo": consulta encendida, cada intervalo procesa lo que llegó
                             # "una_vez": se dejan todos los archivos y la consulta procesa lo pendiente y termina
DIA_DEMO = ST["dia_demo"]    # "auto" o una fecha AAAA-MM-DD de 2025
# -----------------------------------------
spark.conf.set("spark.sql.shuffle.partitions", str(ST["particiones_shuffle"]))
EVID = RAIZ / "docs" / "evidencias" / "tiempo_real"
EVID.mkdir(parents=True, exist_ok=True)
ZONA = ST["zona_horaria"]
VIGENCIA = cfg["clima"]["ffill_max_horas"]
NUMERICAS = list(cfg["features"]["numericas"])
CATEGORICAS = list(cfg["features"]["categoricas"])
RUN_ID = datetime.now(UTC).strftime("%Y%m%dT%H%M%S")
LLEGADAS = f"{ruta('stream_metar')}/run={RUN_ID}"
CHECKPOINTS = f"{ruta('checkpoints')}/alertas/run={RUN_ID}"
SALIDA = ruta("gold_alertas_tr")
MODELOS_DIR = f"{BASE}/{tablas['gold_modelos']}"

modelos = {o: PipelineModel.load(f"{MODELOS_DIR}/final_{o}") for o in ("y_demora15", "y_demora60")}
umbrales = {r.objetivo: r for r in spark.read.json(f"{MODELOS_DIR}/umbrales_final").collect()}
U_AMBAR, U_ROJO = umbrales["y_demora15"].umbral, umbrales["y_demora60"].umbral
VERSION = umbrales["y_demora15"].version
print(f"Corrida {RUN_ID} · modo {MODO} · modelo {VERSION} (ámbar ≥ {U_AMBAR}, rojo ≥ {U_ROJO})")
print("Llegadas:", LLEGADAS)

## 1. Día de la demostración

Si `DIA_DEMO = "auto"`, se elige con los datos de 2025 (test) en el aeropuerto de `streaming.aeropuerto_demo`:

1. horas entre las 6 y las 22 (hora local) con techo bajo o poca visibilidad (IFR/LIFR), tormenta o niebla;
2. solo días con demoras reales por encima del promedio del año, para que las alertas se puedan comparar con lo que
   pasó;
3. se descartan días con menos del 90 % de las salidas de un día típico (feriados largos, huelgas o datos
   incompletos).

También se elige un **día de contraste** sin mal tiempo y con una tasa de demora cercana al promedio: sirve para
mostrar que el sistema no alerta siempre.

In [ ]:
AEROPUERTO = ST["aeropuerto_demo"]
anio_test = int(cfg["periodo"]["fin"][:4])
obs_dia = (
    spark.read.parquet(ruta("silver_metar")).where((F.col("aeropuerto") == AEROPUERTO) & (F.col("year") == anio_test))
    .withColumn("local", F.from_utc_timestamp("obs_utc", ZONA))
    .where(F.hour("local").between(6, 22))
    .groupBy(F.to_date("local").alias("fecha_local"))
    .agg(F.sum(F.col("categoria_vuelo").isin("IFR", "LIFR").cast("int")).alias("obs_ifr_lifr"),
         F.sum("fen_tormenta").alias("obs_tormenta"), F.sum("fen_niebla").alias("obs_niebla"))
    .withColumn("puntaje_clima", F.col("obs_ifr_lifr") + 2 * F.col("obs_tormenta") + F.col("obs_niebla"))
)
salidas_dia = (
    spark.read.parquet(ruta("silver_vuelos"))
    .where(F.col("es_latam") & (F.col("pais") == "BR") & (F.col("origen") == AEROPUERTO) & (F.col("year") == anio_test))
    .where(~F.col("cancelado") & ~F.col("es_atipico") & F.col("demora_salida_min").isNotNull())
    .groupBy("fecha_local").agg(F.count("*").alias("salidas"), F.round(F.avg("y_demora15"), 4).alias("tasa15"),
                                F.round(F.avg("y_demora60"), 4).alias("tasa60"))
)
candidatos = obs_dia.join(salidas_dia, "fecha_local").cache()
mediana = candidatos.approxQuantile("salidas", [0.5], 0.01)[0]
tasa_media = candidatos.agg(F.avg("tasa15")).first()[0]
completos = candidatos.where(F.col("salidas") >= 0.9 * mediana)
top = completos.where(F.col("tasa15") > tasa_media).orderBy(F.desc("puntaje_clima"), F.desc("tasa15")).limit(5).toPandas()
contraste = (completos.where(F.col("puntaje_clima") == 0)
             .orderBy(F.abs(F.col("tasa15") - F.lit(tasa_media))).limit(1).toPandas())
DIA = str(top.fecha_local.iloc[0]) if DIA_DEMO == "auto" else DIA_DEMO
DIA_CONTRASTE = str(contraste.fecha_local.iloc[0]) if len(contraste) else None
top.to_csv(EVID / "candidatos_dia_demo.csv", index=False)
print(f"Salidas típicas por día en {AEROPUERTO}: {mediana:.0f} · tasa media de demora: {tasa_media:.4f}")
print(f"Día de la demostración: {DIA} · día de contraste (sin mal tiempo): {DIA_CONTRASTE}")
top

## 2. Itinerario del día (tabla estática) y archivos de llegada

- **Itinerario:** todas las salidas de LATAM Brasil del día con sus variables de gold, **sin** las de clima (`o_*`,
  `d_*`), que las calcula el flujo. Las etiquetas reales (`y_demora15`, `y_demora60`) se conservan solo para evaluar
  al final; el modelo no las usa.
- **Llegadas:** los METAR originales de los aeropuertos de origen y destino de esos vuelos, desde
  `horas_previas` antes de la primera hora de referencia. Se agrupan en un archivo CSV por hora de observación.

In [ ]:
def itinerario_de(dia):
    gold = spark.read.parquet(ruta("gold_features"))
    return (
        gold.where(F.col("es_latam") & (F.col("pais") == "BR") & (F.col("fecha_local") == F.lit(dia).cast("date")))
        .drop(*[c for c in gold.columns if c.startswith(("o_", "d_"))], "clima_disponible")
    )


itinerario = itinerario_de(DIA).cache()
k_min, k_max = itinerario.agg(F.min(F.unix_timestamp("k_ref")), F.max(F.unix_timestamp("k_ref"))).first()
aeropuertos = [r.a for r in itinerario.select(F.col("origen").alias("a")).union(itinerario.select("destino")).distinct().collect()]
desde_seg = k_min - ST["horas_previas"] * 3600

eventos = (
    spark.read.parquet(ruta("silver_metar")).where(F.col("aeropuerto").isin(aeropuertos))
    .where(F.unix_timestamp("obs_utc").between(desde_seg, k_max))
    .select("aeropuerto", "estacion_iem", F.date_format("obs_utc", "yyyy-MM-dd HH:mm").alias("valid"), "metar",
            F.date_format(F.date_trunc("hour", "obs_utc"), "yyyyMMdd'T'HH").alias("hora"))
    .orderBy("hora", "aeropuerto", "valid").collect()
)
ARCHIVOS = {}
for e in eventos:
    ARCHIVOS.setdefault(e.hora, []).append((e.aeropuerto, e.estacion_iem, e.valid, e.metar))
HORAS = sorted(ARCHIVOS)
print(f"{itinerario.count()} salidas de LATAM Brasil el {DIA} desde {itinerario.select('origen').distinct().count()} aeropuertos")
print(f"{len(eventos)} METAR de {len(aeropuertos)} aeropuertos en {len(HORAS)} archivos horarios ({HORAS[0]} a {HORAS[-1]} UTC)")

In [ ]:
if BUCKET:
    import boto3

    s3 = boto3.client("s3")


def dejar_caer(nombre, filas, carpeta=None):
    """Deja un archivo CSV completo en la carpeta de llegadas (escritura atómica: el flujo nunca lee uno a medias)."""
    texto = io.StringIO()
    w = csv.writer(texto)
    w.writerow(["aeropuerto", "estacion_iem", "valid", "metar"])
    w.writerows(filas)
    clave = f"{tablas['stream_metar']}/run={carpeta or RUN_ID}/{nombre}"
    if BUCKET:
        s3.put_object(Bucket=BUCKET, Key=clave, Body=texto.getvalue().encode("utf-8"))
    else:
        destino = RAIZ_LOCAL / clave
        destino.parent.mkdir(parents=True, exist_ok=True)
        temporal = destino.with_name("." + destino.name)  # Spark ignora los archivos que empiezan con punto
        temporal.write_text(texto.getvalue(), encoding="utf-8")
        temporal.rename(destino)


# El primer archivo se deja antes de iniciar: la carpeta tiene que existir para que el flujo arranque
dejar_caer(f"metar_{HORAS[0]}.csv", ARCHIVOS[HORAS[0]])

## 3. La consulta de streaming

Las cinco piezas de Structured Streaming:

| Pieza | En este notebook |
|---|---|
| Fuente | archivos CSV en la carpeta de llegadas, con **esquema explícito**, **un archivo por micro-lote** (`maxFilesPerTrigger=1`) |
| Consulta | decodificar el METAR (las mismas reglas del notebook 06) y descartar repetidos con **marca de agua** |
| Disparador | `processingTime` en modo continuo o `availableNow` en modo una vez |
| Destino | `foreachBatch`: arma el clima vigente, aplica el modelo y escribe en gold |
| Modo de salida | **append**: cada alerta se emite una sola vez y no cambia |

**Tiempo del evento y marca de agua:** el tiempo que cuenta es la hora de observación del METAR (`obs_utc`), no la hora
en que el archivo llegó. Con una marca de agua de 2 h, un METAR que llega más de 2 h atrasado respecto del más nuevo ya
visto se descarta: para una alerta de T−2 h ese dato ya no sirve (lo corrige el lote mensual). La marca de agua
también permite a Spark borrar del estado las claves viejas de la deduplicación.

**Costo de Lambda:** la decodificación del METAR está escrita dos veces (aquí y en el notebook 06). Se copió idéntica
y se verifica al final comparando las alertas con las predicciones del lote.

In [ ]:
def grupo(columna, patron, i):
    valor = F.regexp_extract(columna, patron, i)
    return F.when(valor != "", valor)


def decodificar(metar):
    """Las mismas reglas del notebook 06: viento, visibilidad, techo, temperatura, presión, fenómenos y categoría."""
    sin_rmk = F.regexp_replace(F.regexp_replace("metar", r"^(METAR|SPECI) ", ""), r" RMK .*$", "")
    metar = metar.withColumn("tiene_tempo", sin_rmk.rlike(r" (TEMPO|BECMG) ").cast("int"))
    metar = metar.withColumn("principal", F.regexp_replace(sin_rmk, r" (TEMPO|BECMG) .*$", ""))
    despues_viento = F.regexp_extract("principal", r"(?:KT|MPS) (.*)$", 1)
    metar = metar.withColumn("cuerpo", F.when(despues_viento != "", despues_viento).otherwise(F.col("principal")))

    PATRON_VIENTO = r" (\d{3}|VRB)(\d{2,3})(?:G(\d{2,3}))?(KT|MPS)"
    a_nudos = F.when(grupo("principal", PATRON_VIENTO, 4) == "MPS", 1.94384).otherwise(1.0)
    metar = (
        metar.withColumn("viento_kt", F.round(grupo("principal", PATRON_VIENTO, 2).cast("double") * a_nudos, 1))
        .withColumn("rafaga_kt", F.round(grupo("principal", PATRON_VIENTO, 3).cast("double") * a_nudos, 1))
    )
    cavok = F.col("cuerpo").rlike(r"(^| )CAVOK( |$)")
    vis_metros = grupo("cuerpo", r"^(?:\d{3}V\d{3} )?(\d{4})(?:NDV)?(?= |$)", 1).cast("int")
    PATRON_SM = r"(?:^| )(M|P)?(?:(\d{1,2}) )?(?:(\d{1,2})/(\d{1,2})|(\d{1,2}))SM(?= |$)"
    entero_solo = grupo("cuerpo", PATRON_SM, 5).cast("double")
    entero = F.coalesce(grupo("cuerpo", PATRON_SM, 2).cast("double"), F.lit(0.0))
    fraccion = grupo("cuerpo", PATRON_SM, 3).cast("double") / grupo("cuerpo", PATRON_SM, 4).cast("double")
    vis_millas = F.coalesce(entero_solo, entero + fraccion)
    metar = metar.withColumn(
        "visibilidad_m",
        F.when(cavok, 10000.0).when(vis_metros == 9999, 10000.0)
        .when(vis_metros.isNotNull(), vis_metros.cast("double")).otherwise(F.round(vis_millas * 1609.34, 0)),
    )
    capas = F.regexp_extract_all("cuerpo", F.lit(r"(?:BKN|OVC|VV)(\d{3})"), 1)
    techo = F.array_min(F.transform(capas, lambda x: x.cast("int"))) * 100
    se_leyo = F.col("viento_kt").isNotNull() | F.col("visibilidad_m").isNotNull()
    metar = metar.withColumn("techo_ft", F.when(F.size(capas) > 0, techo).when(se_leyo | cavok, 99999))
    PATRON_TEMP = r"(?:^| )(M?\d{2})/(M?\d{2})?(?= |$)"
    metar = (
        metar.withColumn("temp_c", F.regexp_replace(grupo("cuerpo", PATRON_TEMP, 1), "M", "-").cast("int"))
        .withColumn("rocio_c", F.regexp_replace(grupo("cuerpo", PATRON_TEMP, 2), "M", "-").cast("int"))
        .withColumn("spread_t_td", F.col("temp_c") - F.col("rocio_c"))
    )
    q = grupo("cuerpo", r"(?:^| )Q(\d{4})(?= |$)", 1).cast("double")
    a = grupo("cuerpo", r"(?:^| )A(\d{4})(?= |$)", 1).cast("double")
    metar = metar.withColumn("presion_hpa", F.round(F.coalesce(q, a / 100 * 33.8639), 1))
    FENOMENOS = {
        "fen_niebla": r"(^| )(VC|MI|BC|PR|FZ)?FG( |$)",
        "fen_neblina": r"(^| )BR( |$)",
        "fen_tormenta": r"(^| )[-+]?(VC)?TS[A-Z]*( |$)",
        "fen_lluvia": r"(^| )[-+]?(VC)?(SH|TS|FZ)?(RA|DZ)[A-Z]*( |$)",
        "fen_nieve": r"(^| )[-+]?(VC|BL|DR|SH)?([A-Z]{2})?SN( |$)",
    }
    for nombre, patron in FENOMENOS.items():
        metar = metar.withColumn(nombre, F.col("cuerpo").rlike(patron).cast("int"))
    millas = F.col("visibilidad_m") / 1609.34
    return metar.withColumn(
        "categoria_vuelo",
        F.when((F.col("techo_ft") < 500) | (millas < 1), "LIFR")
        .when((F.col("techo_ft") < 1000) | (millas < 3), "IFR")
        .when((F.col("techo_ft") <= 3000) | (millas <= 5), "MVFR")
        .when(F.col("techo_ft").isNotNull() & F.col("visibilidad_m").isNotNull(), "VFR"),
    )


COL_OBS = ["aeropuerto", "obs_utc", "viento_kt", "rafaga_kt", "visibilidad_m", "techo_ft", "temp_c", "spread_t_td",
           "presion_hpa", "fen_niebla", "fen_neblina", "fen_tormenta", "fen_lluvia", "fen_nieve", "tiene_tempo",
           "categoria_vuelo"]
esquema_llegada = T.StructType([T.StructField(c, T.StringType()) for c in ("aeropuerto", "estacion_iem", "valid", "metar")])


def flujo_metar(carpeta):
    """Fuente de streaming + decodificación + deduplicación con marca de agua."""
    llegadas = (
        spark.readStream.schema(esquema_llegada).option("header", True).option("maxFilesPerTrigger", 1).csv(carpeta)
        .withColumn("obs_utc", F.to_timestamp("valid", "yyyy-MM-dd HH:mm"))
        .where(F.col("obs_utc").isNotNull() & F.col("metar").isNotNull())
    )
    return (
        decodificar(llegadas).select(*COL_OBS)
        .withWatermark("obs_utc", ST["marca_agua"])
        .dropDuplicates(["aeropuerto", "obs_utc"])
    )

### Qué hace cada micro-lote (`foreachBatch`)

1. Junta los METAR nuevos con el **estado**: las observaciones de las últimas 8 horas por aeropuerto, guardadas en S3
   por número de lote (si el proceso se cae, el lote se repite y se vuelve a escribir igual).
2. El **reloj de eventos** es el METAR más reciente recibido. Toda hora `K` anterior o igual a ese reloj ya tiene su
   clima completo: se puntúan los vuelos con `k_ref` (T−2 h truncado a la hora) en esas horas que aún no se puntuaron.
3. Para cada aeropuerto y hora `K`, el clima vigente es la **última observación anterior a `K`** y con no más de 3 h
   de antigüedad, con la tendencia de presión de 3 h: la misma definición que `clima_horario` en el lote.
4. Aplica los dos modelos guardados (`transform`, nunca `fit`), asigna el nivel y escribe con el número de lote como
   partición: re-ejecutar un lote lo reemplaza, no lo duplica.

In [ ]:
CLIMA_ORIGEN = ["obs_utc", "viento_kt", "rafaga_kt", "visibilidad_m", "techo_ft", "temp_c", "spread_t_td",
                "presion_hpa", "presion_tendencia_3h", "fen_niebla", "fen_neblina", "fen_tormenta", "fen_lluvia",
                "fen_nieve", "tiene_tempo", "horas_desde_obs", "categoria_vuelo"]
CLIMA_DESTINO = ["viento_kt", "visibilidad_m", "techo_ft", "fen_niebla", "fen_tormenta", "fen_lluvia"]
HORA = F.expr("INTERVAL 1 HOUR")


def clima_en(horas, obs):
    """Para cada (aeropuerto, k): la última observación anterior a k con no más de VIGENCIA horas."""
    c = horas.join(obs, "aeropuerto").where(
        (F.col("obs_utc") < F.col("k")) & (F.col("obs_utc") >= F.col("k") - F.expr(f"INTERVAL {VIGENCIA} HOURS")))
    orden = Window.partitionBy("aeropuerto", "k").orderBy(F.desc("obs_utc"))
    return c.withColumn("_n", F.row_number().over(orden)).where("_n = 1").drop("_n")


def clima_vigente(horas, obs):
    tres = F.expr("INTERVAL 3 HOURS")
    antes = (clima_en(horas.withColumn("k", F.col("k") - tres), obs)
             .select("aeropuerto", (F.col("k") + tres).alias("k"), F.col("presion_hpa").alias("_p3")))
    return (
        clima_en(horas, obs).join(antes, ["aeropuerto", "k"], "left")
        .withColumn("presion_tendencia_3h", F.round(F.col("presion_hpa") - F.col("_p3"), 1))
        .withColumn("horas_desde_obs", F.round((F.unix_timestamp("k") - F.unix_timestamp("obs_utc")) / 3600, 2))
        .drop("_p3")
    )


def preparar(df):
    """Las mismas columnas que en el notebook 12, más las que identifican la alerta."""
    return df.select(
        "id_vuelo", "numero_vuelo", "origen", "destino", "salida_prog_utc", "k_ref", "o_obs_utc",
        "o_categoria_vuelo", "y_demora15", "y_demora60",
        *[F.col(c).cast("double").alias(c) for c in NUMERICAS],
        *[F.coalesce(F.col(c).cast("string"), F.lit("desconocido")).alias(f"cat_{c}") for c in CATEGORICAS],
    )


def puntuar(df, modelo, nombre):
    salida = modelo.transform(df).withColumn(nombre, vector_to_array("probability")[1])
    return salida.drop(*[c for c in salida.columns if c not in df.columns and c != nombre])


def leer_o_nada(ruta_tabla):
    try:
        return spark.read.parquet(ruta_tabla)
    except AnalysisException:
        return None


BITACORA = []


def crear_procesador(run, itinerario_run):
    estado_dir = f"{ruta('checkpoints')}/estado_clima/run={run}"

    def procesar(lote, batch_id):
        inicio = time.time()
        nuevas = lote.select(*COL_OBS).persist()
        n_nuevas = nuevas.count()
        registro = {"run": run, "lote": batch_id, "metar_nuevos": n_nuevas, "vuelos_puntuados": 0, "ambar": 0, "rojo": 0}
        estado = leer_o_nada(estado_dir)
        previas = None
        if estado is not None:
            ultimo = estado.where(F.col("lote") < batch_id).agg(F.max("lote")).first()[0]
            previas = None if ultimo is None else estado.where(F.col("lote") == ultimo).drop("lote")
        obs = nuevas if previas is None else previas.unionByName(nuevas)
        reloj = obs.agg(F.max(F.unix_timestamp("obs_utc"))).first()[0]
        if reloj is None:
            nuevas.unpersist()
            BITACORA.append(registro)
            return
        (obs.where(F.unix_timestamp("obs_utc") >= reloj - 8 * 3600).dropDuplicates(["aeropuerto", "obs_utc"])
         .withColumn("lote", F.lit(batch_id)).write.mode("overwrite").partitionBy("lote").parquet(estado_dir))
        obs = spark.read.parquet(estado_dir).where(F.col("lote") == batch_id).drop("lote")
        nuevas.unpersist()

        k_hasta = reloj - reloj % 3600
        salida = leer_o_nada(SALIDA)
        k_desde = None if salida is None else salida.where((F.col("run") == run) & (F.col("lote") < batch_id)) \
            .agg(F.max(F.unix_timestamp("k_ref"))).first()[0]
        pendientes = itinerario_run.where(F.unix_timestamp("k_ref") <= k_hasta)
        if k_desde is not None:
            pendientes = pendientes.where(F.unix_timestamp("k_ref") > k_desde)
        registro["reloj_eventos_utc"] = datetime.fromtimestamp(reloj, UTC).strftime("%Y-%m-%d %H:%M")
        if pendientes.isEmpty():
            BITACORA.append(registro)
            return

        horas = (pendientes.select(F.col("origen").alias("aeropuerto"), F.col("k_ref").alias("k"))
                 .union(pendientes.select(F.col("destino").alias("aeropuerto"), F.col("k_ref").alias("k"))).distinct())
        clima = clima_vigente(horas, obs).cache()
        c_o = clima.select(F.col("aeropuerto").alias("origen"), F.col("k").alias("k_ref"),
                           *[F.col(c).alias(f"o_{c}") for c in CLIMA_ORIGEN])
        c_d = clima.select(F.col("aeropuerto").alias("destino"), F.col("k").alias("k_ref"),
                           *[F.col(c).alias(f"d_{c}") for c in CLIMA_DESTINO])
        vuelos = (pendientes.join(c_o, ["origen", "k_ref"], "left").join(c_d, ["destino", "k_ref"], "left")
                  .withColumn("clima_disponible", F.col("o_obs_utc").isNotNull().cast("int")))
        p = puntuar(puntuar(preparar(vuelos), modelos["y_demora15"], "prob_demora15"), modelos["y_demora60"], "prob_demora60")
        alertas = p.select(
            "id_vuelo", "numero_vuelo", "origen", "destino", "salida_prog_utc", "k_ref", "o_obs_utc", "o_categoria_vuelo",
            F.round("prob_demora15", 5).alias("prob_demora15"), F.round("prob_demora60", 5).alias("prob_demora60"),
            F.when(F.col("prob_demora60") >= U_ROJO, "rojo").when(F.col("prob_demora15") >= U_AMBAR, "ambar")
            .otherwise("verde").alias("nivel_alerta"),
            F.timestamp_seconds(F.lit(reloj)).alias("reloj_eventos_utc"),
            F.round((F.unix_timestamp("salida_prog_utc") - F.lit(reloj)) / 60).alias("minutos_antes_de_salida"),
            F.current_timestamp().alias("emitida_utc"), F.lit(VERSION).alias("version_modelo"),
            "y_demora15", "y_demora60",
            F.lit(run).alias("run"), F.lit(batch_id).alias("lote"),
        )
        alertas.write.mode("overwrite").partitionBy("run", "lote").parquet(SALIDA)
        clima.unpersist()
        escritas = spark.read.parquet(SALIDA).where((F.col("run") == run) & (F.col("lote") == batch_id))
        conteo = {r.nivel_alerta: r["count"] for r in escritas.groupBy("nivel_alerta").count().collect()}
        registro.update(vuelos_puntuados=sum(conteo.values()), ambar=conteo.get("ambar", 0), rojo=conteo.get("rojo", 0),
                        segundos=round(time.time() - inicio, 1))
        BITACORA.append(registro)

    return procesar

In [ ]:
def iniciar_alertas(run, carpeta, checkpoint, modo, itinerario_run):
    escritor = (
        flujo_metar(carpeta).writeStream.queryName(f"alertas_{run.replace('-', '_')}")
        .foreachBatch(crear_procesador(run, itinerario_run))
        .outputMode("append")
        .option("checkpointLocation", checkpoint)
    )
    escritor = (escritor.trigger(processingTime=ST["intervalo_continuo"]) if modo == "continuo"
                else escritor.trigger(availableNow=True))
    return escritor.start()


def iniciar_conteo(carpeta, checkpoint, modo):
    """Segunda consulta: METAR por hora del evento y categoría de vuelo (ventana de 1 h), para el tablero en vivo."""
    conteo = flujo_metar(carpeta).groupBy(F.window("obs_utc", "1 hour").alias("ventana"), "categoria_vuelo").count()
    escritor = (conteo.writeStream.queryName("clima_por_hora").format("memory").outputMode("complete")
                .option("checkpointLocation", checkpoint))
    escritor = (escritor.trigger(processingTime=ST["intervalo_continuo"]) if modo == "continuo"
                else escritor.trigger(availableNow=True))
    return escritor.start()


def ver_bitacora(ultimos=None):
    tabla = pd.DataFrame(BITACORA)
    return tabla if ultimos is None else tabla.tail(ultimos)

## 4. Reproducción del día

- **Modo continuo:** las consultas quedan encendidas; cada `pausa_reproduccion_seg` segundos llega el archivo de la
  hora siguiente y cada `intervalo_continuo` Spark procesa lo nuevo. Las alertas aparecen solas, hora por hora.
- **Modo una vez:** se dejan todos los archivos y la consulta procesa lo pendiente (un archivo por micro-lote) y
  termina. Mismo resultado, sin dejar el clúster encendido.

In [ ]:
if MODO == "una_vez":
    for h in HORAS[1:]:
        dejar_caer(f"metar_{h}.csv", ARCHIVOS[h])

q_alertas = iniciar_alertas(RUN_ID, LLEGADAS, f"{CHECKPOINTS}/alertas", MODO, itinerario)
q_conteo = iniciar_conteo(LLEGADAS, f"{CHECKPOINTS}/conteo", MODO)

if MODO == "continuo":
    for h in HORAS[1:]:
        time.sleep(ST["pausa_reproduccion_seg"])
        dejar_caer(f"metar_{h}.csv", ARCHIVOS[h])
        ultimo = BITACORA[-1] if BITACORA else {}
        print(f"llegó {h} UTC · último lote procesado: {ultimo.get('lote')} · reloj {ultimo.get('reloj_eventos_utc')} · "
              f"puntuados {ultimo.get('vuelos_puntuados')} · ámbar {ultimo.get('ambar')} · rojo {ultimo.get('rojo')}")
    q_alertas.processAllAvailable()
    q_conteo.processAllAvailable()
else:
    q_alertas.awaitTermination()
    q_conteo.awaitTermination()
ver_bitacora()

In [ ]:
# Consulta 2: METAR por hora y categoría (modo complete: la tabla entera se reescribe en cada lote)
spark.sql("SELECT ventana.start AS hora_utc, categoria_vuelo, count FROM clima_por_hora ORDER BY hora_utc, categoria_vuelo") \
    .toPandas().pivot(index="hora_utc", columns="categoria_vuelo", values="count").fillna(0).astype(int)

## 5. Datos que llegan tarde

Se vuelve a enviar el archivo de la primera hora (los mismos METAR reales, reenviados horas después, como pasa cuando
un proveedor repite una entrega). Sus observaciones son más viejas que la marca de agua: Spark las descarta sin tocar
el estado ni las alertas, y lo informa en `numRowsDroppedByWatermark`.

In [ ]:
def descartados(consulta):
    return sum(op.get("numRowsDroppedByWatermark", 0) for p in consulta.recentProgress for op in p.get("stateOperators", []))


antes = descartados(q_alertas) if MODO == "continuo" else 0
dejar_caer(f"reenvio_tardio_{HORAS[0]}.csv", ARCHIVOS[HORAS[0]])
if MODO == "continuo":
    q_alertas.processAllAvailable()
    q_tarde = q_alertas
else:
    q_tarde = iniciar_alertas(RUN_ID, LLEGADAS, f"{CHECKPOINTS}/alertas", "una_vez", itinerario)
    q_tarde.awaitTermination()
print(f"Filas del reenvío: {len(ARCHIVOS[HORAS[0]])} · descartadas por la marca de agua: {descartados(q_tarde) - antes}")
ver_bitacora(2)

## 6. El checkpoint: reiniciar sin perder ni repetir

Se detiene la consulta y se vuelve a lanzar **con el mismo checkpoint**. Spark retoma desde el último archivo
confirmado: como no llegó nada nuevo, no procesa ninguna fila (0 filas de entrada) y no duplica alertas. Es lo que
pasaría si la EC2 se apaga a mitad del día (disponibilidad, sección 2.2 del informe).

In [ ]:
for q in spark.streams.active:
    q.stop()
q_reinicio = iniciar_alertas(RUN_ID, LLEGADAS, f"{CHECKPOINTS}/alertas", "una_vez", itinerario)
q_reinicio.awaitTermination()
filas_reinicio = sum(p["numInputRows"] for p in q_reinicio.recentProgress)
alertas_run = spark.read.parquet(SALIDA).where(F.col("run") == RUN_ID)
print(f"Filas leídas al reiniciar con el mismo checkpoint: {filas_reinicio}")
print(f"Alertas de la corrida: {alertas_run.count()} · vuelos distintos: {alertas_run.select('id_vuelo').distinct().count()} "
      f"· vuelos del itinerario: {itinerario.count()}")

## 7. Reprocesar la historia ("Kappa ligera")

Con un **checkpoint nuevo** y `availableNow`, la misma consulta vuelve a leer la carpeta de llegadas desde el
principio y recalcula todas las alertas. Así se aplicaría una regla o un modelo nuevo a los eventos guardados, sin
un segundo programa. La limitación: solo alcanza hasta donde se guardaron los eventos (aquí, en S3; en Kafka, la
retención, 7 días por defecto).

In [ ]:
RUN_REPLAY = f"{RUN_ID}-replay"
q_replay = iniciar_alertas(RUN_REPLAY, LLEGADAS, f"{CHECKPOINTS}/alertas_replay", "una_vez", itinerario)
q_replay.awaitTermination()
todas = spark.read.parquet(SALIDA).where(F.col("run").isin(RUN_ID, RUN_REPLAY))
comparacion = (
    todas.groupBy("id_vuelo").pivot("run", [RUN_ID, RUN_REPLAY]).agg(F.first("nivel_alerta"))
    .withColumnRenamed(RUN_ID, "en_vivo").withColumnRenamed(RUN_REPLAY, "reprocesado")
)
iguales = comparacion.where(F.col("en_vivo") == F.col("reprocesado")).count()
print(f"Vuelos con el mismo nivel en vivo y reprocesado: {iguales} de {comparacion.count()}")

## 8. ¿Acertaron las alertas? ¿Coinciden con el lote?

- **Contra lo que pasó:** precisión y recall del día con las demoras reales.
- **Contra la capa de lotes:** las dos capas usan el mismo modelo, los mismos umbrales y la misma definición de clima;
  el nivel debería coincidir con `gold/predicciones` (notebook 13). Una diferencia indicaría que la lógica duplicada
  se separó.
- **Anticipación:** minutos entre la emisión (según el reloj de eventos) y la salida programada.

In [ ]:
alertas_run = spark.read.parquet(SALIDA).where(F.col("run") == RUN_ID).cache()
pred_lote = spark.read.parquet(ruta("gold_predicciones")).select("id_vuelo", F.col("nivel_alerta").alias("nivel_lote"),
                                                            F.col("prob_demora15").alias("prob15_lote"))
cruce = alertas_run.join(pred_lote, "id_vuelo", "left")
r = cruce.agg(
    F.count("*").alias("vuelos"),
    F.sum(F.col("nivel_alerta").isin("ambar", "rojo").cast("int")).alias("alertas"),
    F.sum((F.col("nivel_alerta").isin("ambar", "rojo") & (F.col("y_demora15") == 1)).cast("int")).alias("vp15"),
    F.sum(F.col("y_demora15")).alias("demoras15"),
    F.sum((F.col("nivel_alerta") == "rojo").cast("int")).alias("rojas"),
    F.sum(((F.col("nivel_alerta") == "rojo") & (F.col("y_demora60") == 1)).cast("int")).alias("vp60"),
    F.sum(F.col("y_demora60")).alias("demoras60"),
    F.avg((F.col("nivel_alerta") == F.col("nivel_lote")).cast("double")).alias("coincide_lote"),
    F.max(F.abs(F.col("prob_demora15") - F.col("prob15_lote"))).alias("dif_max_prob"),
    F.min("minutos_antes_de_salida").alias("anticipacion_min"),
    F.avg("minutos_antes_de_salida").alias("anticipacion_media"),
).first()
resumen = {
    "dia": DIA, "run": RUN_ID, "modo": MODO, "vuelos": r.vuelos, "alertas_ambar_o_roja": r.alertas,
    "precision_ambar": round(r.vp15 / max(r.alertas, 1), 4), "recall_ambar": round(r.vp15 / max(r.demoras15, 1), 4),
    "alertas_rojas": r.rojas, "precision_roja": round(r.vp60 / max(r.rojas, 1), 4),
    "recall_roja": round(r.vp60 / max(r.demoras60, 1), 4),
    "coincidencia_con_lote": round(r.coincide_lote or 0, 4), "diferencia_max_probabilidad": r.dif_max_prob,
    "anticipacion_minima_min": r.anticipacion_min, "anticipacion_media_min": round(r.anticipacion_media or 0, 1),
    "lotes": len([b for b in BITACORA if b["run"] == RUN_ID]),
}
pd.DataFrame([resumen]).to_csv(EVID / f"resumen_{DIA}.csv", index=False)
ver_bitacora().to_csv(EVID / f"bitacora_{DIA}.csv", index=False)
pd.Series(resumen)

In [ ]:
AZUL, NARANJA, ROJO, GRIS, TINTA = "#2a78d6", "#eb6834", "#c0392b", "#c9c8c2", "#1f1f1f"
por_hora = (
    alertas_run.withColumn("hora_local", F.hour(F.from_utc_timestamp("salida_prog_utc", ZONA)))
    .groupBy("hora_local").agg(
        F.count("*").alias("salidas"),
        F.sum((F.col("nivel_alerta") == "ambar").cast("int")).alias("ambar"),
        F.sum((F.col("nivel_alerta") == "rojo").cast("int")).alias("rojo"),
        F.sum("y_demora15").alias("demoras_reales"),
    ).orderBy("hora_local").toPandas()
)
fig, eje = plt.subplots(figsize=(11, 4.5))
eje.bar(por_hora.hora_local, por_hora.salidas, color=GRIS, label="salidas programadas")
eje.bar(por_hora.hora_local, por_hora.ambar, color=NARANJA, label="alerta ámbar")
eje.bar(por_hora.hora_local, por_hora.rojo, bottom=por_hora.ambar, color=ROJO, label="alerta roja")
eje.plot(por_hora.hora_local, por_hora.demoras_reales, color=AZUL, marker="o", label="salidas con ≥ 15 min (real)")
eje.set_title(f"{DIA}: ¿qué vuelos alertó la capa de velocidad y cuántos se demoraron de verdad? (LATAM Brasil)",
              loc="left", fontsize=10, color=TINTA)
eje.set_xlabel("hora local de salida programada")
eje.spines[["top", "right"]].set_visible(False)
eje.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig(EVID / f"alertas_{DIA}.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
# Lista que recibiría el jefe de turno de Guarulhos: vuelos en alerta, ordenados por hora de salida
(
    alertas_run.where((F.col("origen") == AEROPUERTO) & F.col("nivel_alerta").isin("ambar", "rojo"))
    .select("numero_vuelo", "destino", F.from_utc_timestamp("salida_prog_utc", ZONA).alias("salida_local"),
            "nivel_alerta", "prob_demora15", "prob_demora60", "o_categoria_vuelo", "minutos_antes_de_salida",
            "y_demora15", "y_demora60")
    .orderBy("salida_local").toPandas()
)

## 9. Costo de mantenerlo encendido

| Forma de operar | Horas de cómputo al mes | Cuándo conviene |
|---|---|---|
| Consulta continua (cada 10 min, 24×7) | 720 h de la EC2 encendida | Si el CCO opera las 24 h y la alerta tiene que estar al minuto |
| `availableNow` disparado por la llegada del METAR (cada hora) | solo los minutos de cada micro-lote, más el arranque | Si se acepta el arranque del clúster en cada hora |
| Lote nocturno | ~1 h por día | No sirve para T−2 h: no ve el clima del día |

El precio por hora de la instancia se consulta en la página de precios de EC2 para `us-east-1` (no se escribe aquí
para no dejar una cifra desactualizada). La decisión de negocio es comparar ese costo con el valor que el notebook 15
asigna a avisar a tiempo.

In [ ]:
for q in spark.streams.active:
    q.stop()
spark.stop()